In [75]:
! pip install pandas openpyxl

import pandas as pd
import sys

In [76]:
from datetime import datetime, timezone

def epoch_to_str(epoch_ts, use_utc=True):
    try:
        ts = float(epoch_ts)
        dt = datetime.fromtimestamp(ts, tz=timezone.utc) if use_utc else datetime.fromtimestamp(ts)
        return f"{dt.month}/{dt.day}/{dt.year} {dt:%H:%M}"
    except (TypeError, ValueError, OSError):
        return None

## Import the labeled file

In [77]:
labeled_file = pd.read_excel("labeling_final.xlsx")
comment_info_file = pd.read_csv("breach_post_comments.csv")
post_info_file = pd.read_csv("final_breach_posts.csv")
new_rows = []

# Every row is going to be a post or a comment, so let's start by checking which one it is
for row in labeled_file.itertuples():
    row_id = row[2]
    if row[4] == "Comment":
        result = comment_info_file[comment_info_file["comment_link"].astype(str).str.strip() == str(row_id).strip()]
        if result.empty:
            print(f"No comment found for id: {row_id}")
        else:
            row_values = result.iloc[0].to_dict()
            new_rows.append({
                "file_name": row[1],
                "id": row_id,
                "timestamp": epoch_to_str(row_values.get("time_of_comment")),
                "author": row_values.get("author"),
                "subreddit": row_values.get("subreddit"),
                "title": "",
                "content": row[3],
                "post_comment": "Comment",
                "persona": row[5],
                "relationship": row[6],
                "engagement": row[7],
                "irrelevant": row[8],
                "upvotes": row_values.get("num_upvotes"),
                "downvotes": row_values.get("num_downvotes"),
                "permalink": row_values.get("link")
            })
    elif row[4] == "Post":
        result = post_info_file[post_info_file["Post ID"].astype(str).str.strip() == str(row_id).strip()]
        if result.empty:
            print(f"No post found for id: {row_id}")
        else:
            row_values = result.iloc[0].to_dict()
            new_rows.append({
                "file_name": row[1],
                "id": row_id,
                "timestamp": row_values.get("Post Time (UTC)"),
                "author": row_values.get("Author"),
                "subreddit": row_values.get("Subreddit"),
                "title": row_values.get("Post Title"),
                "content": row[3],
                "post_comment": "Post",
                "persona": row[5],
                "relationship": row[6],
                "engagement": row[7],
                "irrelevant": row[8],
                "upvotes": row_values.get("No. Upvotes"),
                "downvotes": row_values.get("No. Downvotes"),
                "permalink": row_values.get("Link")
            })
    else:
        print(f"Unlabeled ID {row_id} not found!")
        sys.exit(0)

res = pd.DataFrame(new_rows)

In [78]:
# Remove post 55

id_str = res["file_name"].fillna("").astype(str).str.strip()
res = res[~id_str.str.startswith("55")]

# Replace "unsure" persona labels
res["persona"] = res["persona"].fillna("").astype(str).str.strip()
unclear_mask = res["persona"].str.casefold().eq("unclear")
res.loc[unclear_mask, "persona"] = "General observser"
print(res[res["persona"].str.casefold().eq("unclear")][["file_name", "id", "persona"]].head(20))
print("Remaining unclear count:", res["persona"].str.casefold().eq("unclear").sum())

Empty DataFrame
Columns: [file_name, id, persona]
Index: []
Remaining unclear count: 0


In [79]:
# Check all data is there

from pathlib import Path

comments_dir = Path("comments")

# Files physically present in comments/
comment_files = {p.name for p in comments_dir.glob("*.txt")}

# Names in res
res_files_raw = (
    res["file_name"]
    .dropna()
    .astype(str)
    .str.strip()
)
res_files_raw = {x for x in res_files_raw if x}

# Normalize res names to .txt (handles values like "1_2" vs "1_2.txt")
def to_txt(name: str) -> str:
    return name if name.lower().endswith(".txt") else f"{name}.txt"

res_files = {to_txt(x) for x in res_files_raw}

# Set differences
in_res_not_in_comments = sorted(res_files - comment_files)
in_comments_not_in_res = sorted(comment_files - res_files)

print(f"res file_name entries (normalized): {len(res_files)}")
print(f"files in comments/: {len(comment_files)}")
print(f"Missing in comments/: {len(in_res_not_in_comments)}")
print(f"Missing in res: {len(in_comments_not_in_res)}")

if in_res_not_in_comments:
    print("\nIn res but not in comments/ (first 30):")
    print(in_res_not_in_comments[:30])

if in_comments_not_in_res:
    print("\nIn comments/ but not in res (first 30):")
    print(in_comments_not_in_res[:30])

res file_name entries (normalized): 1008
files in comments/: 1008
Missing in comments/: 0
Missing in res: 0


In [80]:
# Check that all rows are labeled

def to_bool_like(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y", "t"}

irrelevant_true = res["irrelevant"].apply(to_bool_like)

persona_filled = res["persona"].notna() & res["persona"].astype(str).str.strip().ne("")
relationship_filled = res["relationship"].notna() & res["relationship"].astype(str).str.strip().ne("")
engagement_filled = res["engagement"].notna() & res["engagement"].astype(str).str.strip().ne("")

valid_mask = irrelevant_true | (persona_filled & relationship_filled & engagement_filled)
invalid_rows = res[~valid_mask]

print(f"Total rows checked: {len(res)}")
print(f"Valid rows: {valid_mask.sum()}")
print(f"Invalid rows: {len(invalid_rows)}")

if not invalid_rows.empty:
    print("\nRows failing rule (first 20):")
    print(
        invalid_rows[
            ["file_name", "id", "irrelevant", "persona", "relationship", "engagement"]
        ].head(20).to_string(index=False)
    )

Total rows checked: 1008
Valid rows: 1008
Invalid rows: 0


In [81]:
# Check consistency of persona/relationship for repeated authors

tmp = res.copy()

for col in ["author", "persona", "relationship"]:
    tmp[col] = tmp[col].fillna("").astype(str).str.strip()

# Ignore blank authors
tmp = tmp[tmp["author"] != ""]

# Inconsistency check for repeated authors:
# Only persona + relationship matter (engagement differences are ignored).

def non_empty_unique_count(series):
    vals = [v for v in series if v != ""]
    return len(set(vals))

author_consistency = (
    tmp.groupby("author", as_index=False)
       .agg(
           persona_unique=("persona", non_empty_unique_count),
           relationship_unique=("relationship", non_empty_unique_count),
           row_count=("author", "size")
       )
)

inconsistent_authors = author_consistency[
    (author_consistency["persona_unique"] > 1) |
    (author_consistency["relationship_unique"] > 1)
].sort_values(["persona_unique", "relationship_unique", "row_count"], ascending=False)

print(f"Total authors checked: {len(author_consistency)}")
print(f"Inconsistent authors: {len(inconsistent_authors)}")

from pathlib import Path

# Write inconsistencies to CSV instead of TXT
report_csv_path = Path("author_persona_relationship_differences.csv")

if not inconsistent_authors.empty:
    bad_author_set = set(inconsistent_authors["author"])

    bad_rows = tmp[tmp["author"].isin(bad_author_set)][
        ["author", "file_name", "id", "persona", "relationship", "engagement", "irrelevant"]
    ].sort_values(["author", "id"], kind="stable")

    bad_rows.to_csv(report_csv_path, index=False, encoding="utf-8")
    print(f"Differences found. CSV written to: {report_csv_path}")
else:
    print("No persona/relationship differences found across repeated authors.")

Total authors checked: 514
Inconsistent authors: 86
Differences found. CSV written to: author_persona_relationship_differences.csv


In [82]:
res.to_csv("labeled_data_with_info.csv", index=False, encoding="utf-8")